# Rapport du projet P2I 7 — Sujet 7-b

Ce notebook présente la démarche du projet et un exemple de calcul sur les données du dépôt.

La copie de présentation adapte les chemins et le bloc de résolution au dépôt GitHub. Les extraits de factorisation sont des exemples commentés. Des précisions signalent les limites de la Jacobienne numérique, du stockage dense et des unités. Le rapport original est conservé dans les références locales.

Pour lancer le calcul, installer les dépendances du dépôt puis exécuter les cellules. L'interface graphique reste optionnelle dans le bloc de résolution.


## 1. Présentation générale du projet et Enjeux d'Ingénierie

### 1.1 Contexte urbain et architecture maillée
L'accès à l'eau potable constitue un pilier fondamental des infrastructures urbaines modernes. Pour répondre à la demande fluctuante des particuliers et des industries, les municipalités déploient des réseaux de distribution complexes. Afin de garantir la continuité du service, notamment pour permettre la maintenance locale d'une canalisation sans interrompre l'alimentation des quartiers adjacents, ces réseaux adoptent une **architecture maillée**. 

Si cette redondance structurelle offre une excellente résilience opérationnelle, elle complexifie considérablement la détermination analytique des grandeurs physiques (débits et pressions). Contrairement à un réseau en antenne (arborescent), les fluides circulent ici par de multiples chemins interconnectés, rendant obligatoire l'usage de la modélisation numérique pour l'ingénieur en charge du dimensionnement.

### 1.2 Objectifs scientifiques et cahier des charges

Le projet vise à calculer les charges hydrauliques, les débits et les pressions d'un réseau maillé, puis à explorer ses paramètres de dimensionnement.

La plage de 2 à 6 bars évoquée dans le rapport est une consigne du projet. L'interface et le code calculent et affichent des pressions en atmosphères. Une validation physique doit distinguer ces unités ; cette étude pédagogique ne constitue pas une certification de conformité.


### 1.3 Démarche de modélisation et structure du Notebook
Pour guider le lecteur à travers notre démarche d'ingénierie, ce notebook est articulé selon la progression suivante :
* **Le modèle linéaire :** Une première approche pédagogique simplifiée où les pertes de charge sont supposées proportionnelles au débit, permettant de valider l'assemblage matriciel de nos structures de données.
* **Le modèle non-linéaire réel :** Introduction des lois de la mécanique des fluides réels, formulant un système d'équations non-linéaires résolu par la méthode itérative de **Newton-Raphson**.
* **L'optimisation algorithmique ($LDL^T$) :** Développement d'un solveur exploitant les propriétés physiques de symétrie et de compacité de la matrice Jacobienne pour s'affranchir de la "boîte noire" de la méthode `numpy.linalg.solve`.
* **Analyse critique et Outil d'aide à la décision :** Confrontation de nos simulations aux contraintes géographiques du grand réseau (impact majeur de l'altitude) et présentation de notre application interactive permettant d'évaluer en temps réel des scénarios de réaménagement.

---

## 2. Démarche scientifique

### 2.1 : Point de départ

Nous avons à notre disposition, pour deux réseaux différents : 
- Les caractéristiques matérielles du réseau (position des nœuds et des canalisations, diamètre des canalisations, position de la conduite d'adduction)
- Les conditions aux limites / débits de fuite à chaque noeud du réseau

Le “grand réseau” est représenté sur la figure suivante : 


![Représentation du grand réseau issue du projet](docs/Grand_reseau.png)


### 2.2 : Hypothèses

Afin de modéliser un réseau de distribution d’eau de manière exploitable numériquement, plusieurs hypothèses physiques et mathématiques sont nécessaires. Sans ces simplifications, le problème relèverait des équations complètes de Navier-Stokes, dont la résolution sur un réseau entier est irréaliste en pratique. Le modèle adopté cherche donc à conserver les phénomènes hydrauliques essentiels tout en restant calculable efficacement. 

L’eau est supposée incompressible et indilatable. En effet, dans les conditions usuelles de pression et de température d’un réseau d’eau potable, les variations de volume sont négligeables. On peut donc assimiler la masse volumique de l’eau à une constante dans tout le réseau. Cette hypothèse permet notamment d’appliquer directement la conservation du débit aux nœuds. 

Le régime est supposé stationnaire : les débits, pressions et potentiels hydrauliques ne dépendent pas du temps. Les phénomènes transitoires (mise en eau, ouverture de vannes, coups de bélier, etc.) sont donc négligés. Cette hypothèse simplifie fortement le problème puisqu’elle permet d’obtenir un système algébrique plutôt qu’un système différentiel dépendant du temps.

L’écoulement est considéré turbulent dans les canalisations, ce qui correspond au comportement réel des réseaux d’eau sous pression. Les pertes d’énergie dues aux frottements sont alors modélisées par la relation empirique de Hazen et Williams : \
$$J = \frac{\Delta H}{L} = 10.69 \frac{|Q|^{1.85}}{C_{HW}^{1.85} D^{4.87}} \tag{1}$$
Avec :
- $J$ : perte de charge par unité de longueur (sans dimension),
- $\Delta H$ : différence de charge hydraulique (en $m$),
- $L$ : longueur de la conduite (en $m$),
- $Q$ : débit volumique (en $m^3 \ s^{-1}$),
- $C_{HW}$ : coefficient de rugosité du matériau des conduites, fixé à $100$ pour la fonte (sans dimension)
- $D$ : diamètre hydraulique de la conduite (en $m$).

Les pertes de charge singulières (coudes, vannes, bifurcations, rétrécissements…) sont négligées devant les pertes de charge linéaires. 

Par ailleurs, on suppose que l’écoulement dans chaque canalisation peut être décrit par une vitesse moyenne uniforme sur toute la section $S$ :
$$ V = \frac{Q}{S} \tag{2}$$

Enfin, chaque canalisation est discrétisée en plusieurs tronçons de même longueur et de même diamètre, définis à partir des données du réseau. Cette discrétisation permet d’obtenir une estimation plus précise des pressions et potentiels le long des conduites, sans complexifier le modèle.

### 2.3 : Lois et théorèmes utilisés

Le modèle repose principalement sur deux lois physiques. 

La première est la conservation du débit aux nœuds, analogue hydraulique de la loi des nœuds de Kirchhoff ou du théorème de Millman. Pour chaque nœud du réseau : 
$$ \Sigma Q_{entrant} - \Sigma Q_{sortant}  = Q_{Débit \ de \ fuite} \tag{3}$$

Dans le réseau complet, la masse d’eau est conservée : 
$$ \Sigma Q_{Débit \ de \ fuite} = - Q_{Entrant \ au \ noeud \ d'adduction} \tag{4}$$

La seconde est l’équation de Bernoulli, utilisée sous forme énergétique, qui nous permet de calculer les pressions à partir des potentiels hydrauliques obtenus numériquement : 
$$H = \frac{P}{\rho g} + \frac{V^2}{2g} + z \tag{5}$$
Dans laquelle :
- $H$ est la charge hydraulique ou potentiel hydraulique (en $m$),
- $z$ l’altitude (en $m$),
- $P$ la pression (en $Pa$),
- $ρ$ la masse volumique de l’eau (en $kg \ m^{-3}$),
- $g$ l’accélération de la pesanteur (en $m \ s^{-2}$),
- $V$ la vitesse moyenne de l’écoulement (en $m \ s^{-1}$).


### 2.4 : Etapes de résolution

Cette section présente la formulation des équations de conservation et leur résolution. Pour N nœuds, les N-1 charges inconnues sont regroupées dans un vecteur ; la charge H0 du nœud source est imposée par les paramètres de l'étude.


### A/ Modèle linéaire : Formulation matricielle et assemblage de la matrice de conductance

Dans le modèle linéaire, la perte de charge subie par le fluide est supposée proportionnelle au débit injecté. Pour une canalisation reliant un nœud $i$ à un nœud $j$, le débit circulant $Q_{ji}$ s'exprime par la relation linéaire :
$$Q_{ji} = Y_{ji} \cdot (H_j - H_i) \tag{6}$$
Où $Y_{ji} = \frac{1}{K_{ji}}$ représente **l'admittance hydraulique** de la conduite, inverse de la **résistance hydraulique** $K$.

#### Application de la loi des nœuds (Conservation de la masse)

En chaque nœud $i$ (avec $i \in \{1, N-1\}$), le principe de conservation de la masse, analogue à la loi des nœuds de Kirchhoff, impose que la somme des débits entrants et sortants via les canalisations voisines soit égale au débit de fuite ou de consommation imposé, noté $Q_{fuite, i}$ :
$$\sum_{j \in \mathcal{V}(i)} Q_{ji} = Q_{fuite, i} \tag{3}$$
Où $\mathcal{V}(i)$ représente l'ensemble des canalisations adjacentes au nœud $i$. En substituant la loi de comportement linéaire $ \left( (3) + (6) \right) $, on obtient :
$$\sum_{j \in \mathcal{V}(i)} Y_{ji} \cdot (H_j - H_i) = Q_{fuite, i} \tag{7}$$

#### Algorithme de remplissage et d'assemblage matriciel

Pour résoudre le système du réseau complet à $N$ noeuds, cette équation est réécrite sous la forme d'un système linéaire compact :
$$\mathbf{M} \cdot H = Q_{net} \tag{8}$$

Le remplissage de la matrice de conductance $\mathbf{M}$ (de dimension $(N-1) \times (N-1)$) et du vecteur second membre des débits "nettoyés" $Q_{net}$ s'effectue par une analyse nodale systématique implémentée dans la méthode `calcul_potentiels_lineaire` :

1. **Termes diagonaux ($M_{ii}$) :** Ils regroupent l'impact du potentiel du nœud $i$ sur ses propres flux. En isolant $H_i$, on constate qu'il est multiplié par la somme des admittances de toutes les conduites connectées à ce nœud :
   $$M_{ii} = -\sum_{j \in \mathcal{V}(i)} Y_{ji} \tag{9}$$

2. **Termes extra-diagonaux ($M_{ij}$ pour $j \neq i$) :** Ils traduisent l'influence mutuelle entre le nœud $i$ et son voisin $j$. Si une canalisation existe entre $i$ et $j$, l'admittance est ajoutée :
   $$M_{ij} \ += Y_{ji}$$
   *Si aucune canalisation ne relie directement $i$ et $j$, le coefficient reste nul : $M_{ij} = 0$, conférant à $\mathbf{M}$ une structure creuse.*

3. **Traitement de la condition aux limites $H_0$ :** Le nœud $0$ possède un potentiel imposé et connu $H_0$. Il ne fait donc pas partie des inconnues du vecteur $H$. Lors du parcours des voisins d'un noeud $i$, si $j = 0$, le terme correspondant $Y_{0i} \cdot H_0$ est un produit connu. Il est alors basculé à droite de l'égalité dans le vecteur second membre $Q_{net}$ :
   $$Q_{net}[i-1] = Q_{fuite, i} - Y_{0i} \cdot H_0 \tag{10}$$

#### Résolution du système matriciel

Une fois la matrice assemblée, le système d'équations linéaires à résoudre s'écrit sous la forme :
$$\mathbf{M} \cdot H = Q_{net} \tag{8}$$

Pour des raisons d'efficacité numérique, ce système est résolu directement sans jamais calculer explicitement de matrice inverse. En pratique, nous utilisons soit la fonction générique `np.linalg.solve(M, Q)`, soit notre solveur optimisé exploitant la décomposition $LDL^T$ `resoudre_systeme_symetrique_numpy(M, Q)`.

### B/ Modèle non-linéaire : Résolution par Newton-Raphson et optimisation $LDL^T$

Le modèle réel obéit à la loi empirique de **Hazen-Williams**, introduisant une non-linéarité forte (puissance non entière). Le débit circulant s'exprime désormais par :
$$Q_{ji}(H) = \text{sgn}(H_j - H_i) \cdot 100 \cdot \left( \frac{D_{ji}^{4.87} \cdot |H_j - H_i|}{10.69 \cdot L_{ji}} \right)^{\frac{1}{1.85}} \tag{11}$$

En combinant $(3) \text{ et } (11)$, le système d'équations de conservation devient un système non linéaire d'équations résiduelles : $F(H) = 0$, où chaque composante $I$ s'écrit :
$$F_i(H) = \left( \sum_{j \in \mathcal{V}(i)} Q_{ji}(H) \right) - Q_{fuite, i} = 0 \tag{12}$$

Le système ne peut donc plus être résolu comme un système linéaire, imposant le recours à une méthode itérative de linéarisation locale : Newton-Raphson.

#### Algorithme itératif de Newton-Raphson

Pour converger vers la solution physique, nous implémentons l'algorithme de Newton-Raphson. À partir d'une estimation initiale $H^{(0)}$ (générée par un profil de décroissance linéaire à partir de $H_0$ afin d'éviter les cas où $\Delta H = 0$ qui augmentent le nombre d'itérations nécessaires), chaque itération $k$ calcule une correction $\delta H$ en résolvant le système linéaire tangent associé à la **matrice Jacobienne** $\mathbf{J}$ :
$$\mathbf{J}\left(H^{(k)}\right) \cdot \delta H = - F\left(H^{(k)}\right) \tag{13}$$
Le vecteur des potentiels est ensuite actualisé jusqu'à ce que la norme de la correction $||\delta H||$ devienne inférieure à la tolérance requise (paramètre `precision` proposé arbitrairement à $10^{-10}$) : 
$$H^{(k+1)} = H^{(k)} + \delta H \tag{14}$$

#### Remplissage numérique de la Jacobienne

Les éléments de la matrice Jacobienne correspondent aux dérivées partielles des fonctions résiduelles par rapport aux potentiels inconnus : $J_{mn} = \frac{\partial F_m}{\partial H_n}$. 
Pour faciliter l'adaptation du code face à d'éventuelles modifications des lois de perte de charge, notre fonction `jacobienne_numerique` réalise une **différenciation numérique par différences finies en avant** avec un incrément infinitésimal $\epsilon = 10^{-8}$ :

* **Sensibilité au potentiel propre ($J_{ii}$) :** La dérivée par rapport à $H_i$ est évaluée en perturbant localement le potentiel du nœud courant :
  $$J_{ii} \approx \frac{F_i(H_1, \dots, H_i + \epsilon, \dots, H_{N-1}) - F_i(H)}{\epsilon} \tag{15}$$
* **Sensibilité aux potentiels adjacents ($J_{ij}$) :** Pour un nœud voisin $j \neq 0$, la variation croisée est calculée de la même manière :
  $$J_{ij} \approx \frac{F_i(H_1, \dots, H_j + \epsilon, \dots, H_{N-1}) - F_i(H)}{\epsilon} \tag{16}$$

#### Optimisation algorithmique par décomposition $LDL^T$

Le modèle de réseau non orienté conduit à des relations de symétrie entre les dérivées croisées. La différenciation numérique peut introduire des écarts à cette symétrie exacte.

Le solveur développé utilise une factorisation $A = LDL^T$, puis trois résolutions : descente triangulaire, résolution diagonale et remontée triangulaire. Il évite de former explicitement une matrice inverse.

La version du dépôt stocke les matrices sous forme de tableaux denses et ne réalise pas de pivotement. La structure physique du réseau peut suggérer des zéros, sans que l'implémentation exploite un stockage creux. Une factorisation dense reste d'ordre cubique ; aucun avantage asymptotique ou gain de temps général n'est déduit de la seule utilisation de LDLᵀ.

Les formules et les étapes sont détaillées en annexe.


### 2.5 : Critiques

#### A/ Limites liées aux hypothèses du modèle

Le modèle développé repose sur plusieurs hypothèses simplificatrices nécessaires à la modélisation numérique du réseau hydraulique, mais qui limitent également la précision des résultats obtenus.

Tout d’abord, la loi de Hazen-Williams réduit notre résolution aux régimes turbulents. Notre programme ne calcule pas le nombre de Reynolds : si Re < 2300, le régime est laminaire, et si Re > 4000 il est turbulent, avec une zone de transition entre les deux. Un réseau d’eau ne peut pas présenter d'écoulements laminaires mais certaines conduites avec un faible débit pourraient être dans la zone de transition, rendant la loi de Hazen-Williams imprécise. Utiliser les formules de Darcy-Weisbach et de Colebrook-White pour prendre en compte et calculer le coefficient de Reynold serait de ce fait plus fiable.

De plus, dans un réseau réel, le coefficient de rugosité $ C_{HW} $ dépend non seulement du matériau, mais aussi de l’âge et de l’état des canalisations. 

Enfin, les consommations aux nœuds sont considérées constantes dans le cadre du régime stationnaire. En pratique, elles varient selon l’usage des utilisateurs. 



#### B/ Vérifications mises en place

Le contrôle principal reconstitue le débit de la source à partir des charges calculées et le compare à la somme des consommations imposées. Il vérifie la conservation globale du débit sur le réseau étudié.

Ce bilan est nécessaire mais ne valide pas à lui seul toutes les hypothèses physiques, les unités ou la précision de chaque pression locale. Les tests du dépôt ajoutent notamment une comparaison du solveur LDLᵀ à NumPy et un cas linéaire de solution analytique connue.


#### C/ Choix de la Jacobienne numérique

La Jacobienne par différences finies permet de réutiliser les fonctions de perte de charge, au prix d'une erreur d'approximation et d'un choix d'incrément numérique.

Une Jacobienne analytique constitue une piste pour réduire le coût et contrôler les dérivées plus précisément. Elle resterait soumise aux erreurs d'arrondi de l'arithmétique flottante. La version du dépôt conserve le calcul numérique avec un incrément de 1e-8.


#### D/ Cohérence des résultats

Le rapport décrit des écarts importants entre les pressions obtenues aux points hauts et bas du réseau. Les variations d'altitude interviennent dans la relation de Bernoulli.

Dans les configurations explorées pendant le projet, modifier les diamètres, la charge source et le maillage n'a pas permis de satisfaire simultanément les consignes avec une seule adduction. Ce constat porte sur les scénarios étudiés ; il ne constitue pas une preuve générale d'impossibilité.

Le rapport propose d'étudier une deuxième adduction comme prolongement. Cette piste n'est pas implémentée ni validée dans le logiciel présenté ici.


## 3. Résultats de la simulation

### 3.1 : Synthèse de nos fichiers

Les données et les modules nécessaires sont fournis dans le dépôt :

- [Coordonnées des nœuds](noeuds2.txt)
- [Connexions entre les nœuds](connec2.txt)
- [Consommations aux nœuds](debit2.txt)
- [Diamètres des canalisations](Diam2.txt)
- [Contrôleur de résolution](Main_tout_modele.py)
- [Classes du modèle](Classes_tout_modele.py)
- [Interface graphique](Affichage_tout_modele.py)
- [Solveur LDLᵀ](Inversion_matrice_symetrique.py)

Le [format des données](docs/donnees.md) précise les unités et les conversions utilisées.


### 3.2 : Programme principal

#### A/ Mode d'emploi

Notre code intègre une interface graphique développée avec la bibliothèque `Tkinter`. Conçue comme un véritable outil d'aide à la décision pour ingénieur hydraulicien, elle permet de s'affranchir des lignes de code pour interagir directement, visuellement et en temps réel avec le réseau modélisé. 

Voici le guide utilisateur exhaustif pour exploiter l'ensemble des fonctionnalités de l'interface :

#### Visualisation thématique globale
Dans le panneau de contrôle latéral, un groupe de boutons d'option (*Radiobuttons*) vous permet de modifier instantanément la cartographie du réseau affiché sur le canevas :
* **Mode "Aucun" / Neutre :** Affiche la structure brute du réseau.
* **Mode "Potentiels $H$" :** Applique un gradient de couleur sur les nœuds et canalisations en fonction de leur charge hydraulique en mètres.
* **Mode "Pressions $P$" :** Colore les éléments selon leur pression locale calculée. C'est le mode visuel privilégié pour repérer instantanément les anomalies de fonctionnement hydraulique.
* **Mode "Débits $Q$" :** Modifie la coloration des canalisations en fonction de l'intensité du flux volumique qui les traverse, et indique par une flèche le sens de ce dernier.

#### Inspection locale et Analyse des données
L'interface offre une interactivité dynamique pour analyser précisément chaque composant sans encombrer l'écran :
* **Le survol de la souris :** En passant simplement le curseur sur un nœud ou sur une canalisation, une infobulle contextuelle (*tooltip*) apparaît instantanément. Elle détaille les caractéristiques de l'élément et ses variables d'état calculées.
* **Le clic gauche :** Sélectionne un élément spécifique pour verrouiller l'affichage de ses propriétés dans le panneau d'édition ou pour préparer une modification structurelle.

#### Interaction et Modification dynamique des paramètres
Vous pouvez tester instantanément l'impact de décisions d'ingénierie sur le réseau grâce aux réglages :
* **Ajustement de la source ($H_0$) :** Variez la charge de la conduite d'adduction principale pour simuler l'augmentation ou la diminution de puissance d'une station de pompage en amont.
* **Modification de la précision des calculs :** Changez le nombre de tronçons utilisé pour discrétiser les canalisations afin d'affiner au mieux la précision des potentiels et pressions calculées en fonction du besoin.
* **Modification des caractéristiques d'un objet :** Modifiez dynamiquement les diamètres des conduites $D$ ou forcez une augmentation locale des débits de fuite imposés aux nœuds (pour simuler un pic de consommation urbaine ou une rupture de canalisation par exemple).

Après validation d'une modification, l'interface relance les calculs et actualise l'affichage. Le solveur utilisé est celui choisi pour le contrôleur. Ce recalcul est synchrone dans la version actuelle.

#### Modification structurelle du réseau
Pour éprouver la résilience de notre architecture maillée face à un accident majeur, l'interface intègre une fonction de modification topologique :
* Sélectionnez une canalisation par un clic, puis utilisez le bouton **"Supprimer la canalisation"**. 
* L'interface reconstruit immédiatement les matrices d'admittance, élimine la branche du système d'équations et recalcule le nouvel équilibre des fluides. Cela permet de visualiser comment le débit se réorganise automatiquement par les autres mailles pour continuer à alimenter le reste de la ville.

#### Contrôle de conformité automatique
Pour réaliser un diagnostic rapide de la conformité, l'interface propose un affichage binaire lors de la sélection du **Mode "Pression $P$"** :
* Tous les nœuds sont colorés en vert, à l'exception de ceux dont la pression sort de la plage choisie pour l'étude de sécurité ($P < P_{min}$ ou $P > P_{max}$) qui apparaissent en rouge.
* L'interface permet également la modification interactive des consignes $P_{min}$ et $P_{max}$ et la mise à jour dynamique de l'affichage des nœuds.
* Cet outil permet de valider d'un seul coup d'œil si les modifications géométriques apportées aux diamètres ou à la charge $H_0$ ont permis de stabiliser le réseau.

Voici un exemple d'affichage : 

![Capture originale de l'interface du P2I](docs/Exemple_affichage.png)


#### B/ Résolution

In [1]:
from pathlib import Path
import sys
from time import perf_counter
import numpy as np

# Retrouver la racine si le notebook est ouvert depuis un sous-dossier.
racine = Path.cwd().resolve()
while not (racine / "Main_tout_modele.py").is_file():
    if racine.parent == racine:
        raise FileNotFoundError("Ouvrir le notebook depuis le dossier du dépôt.")
    racine = racine.parent
if str(racine) not in sys.path:
    sys.path.insert(0, str(racine))

from Main_tout_modele import Controleur

reseau = [str(racine / nom) for nom in ("connec2.txt", "noeuds2.txt", "debit2.txt", "Diam2.txt")]
solutions = {}
for methode in ("LDLT", "Numpy"):
    controleur = Controleur(reseau, "Non Linéaire", methode, 1e-8, 200, 2, 9)
    debut = perf_counter()
    controleur.resolution()
    duree = perf_counter() - debut
    solutions[methode] = controleur
    print(f"{methode} : {len(controleur.noeuds)} nœuds, {len(controleur.canalisations)} canalisations ; un calcul en {duree:.6f} s")

ecart = np.max(np.abs(np.asarray(solutions["LDLT"].potentiels) - np.asarray(solutions["Numpy"].potentiels)))
print(f"Écart maximal entre les charges calculées : {ecart:.3e} m")
print("Les durées ci-dessus correspondent à un lancement unique, sans valeur de benchmark statistique.")

AFFICHER_INTERFACE = False
if AFFICHER_INTERFACE:
    from Affichage_tout_modele import Affichage_reseau
    Affichage_reseau(solutions["LDLT"]).mainloop()


Cohérence du modèle : -Qf0 = 672.0000000000002 = somme(debits) = 672
LDLT : 30 nœuds, 50 canalisations ; un calcul en 0.024735 s
Cohérence du modèle : -Qf0 = 671.9999999999999 = somme(debits) = 672
Numpy : 30 nœuds, 50 canalisations ; un calcul en 0.016401 s
Écart maximal entre les charges calculées : 2.842e-14 m
Les durées ci-dessus correspondent à un lancement unique, sans valeur de benchmark statistique.


## 4. Annexe

### 4.1 : Détails mathématiques et algorithmiques de la décomposition $LDL^T$

### A/ Principe de la factorisation

Pour une matrice carrée réelle symétrique dont les pivots successifs permettent une factorisation sans pivotement, la décomposition s'écrit :

$$A = L D L^T.$$

$L$ est triangulaire inférieure à diagonale unitaire ; $D$ est diagonale. Les termes de $D$ ne sont pas supposés positifs pour les systèmes traités ici. La version du programme lève une erreur lorsqu'un pivot est nul.


### B/ Formules de récurrence analytiques

En développant le produit matriciel élément par élément ($\mathbf{A} = \mathbf{L}\mathbf{D}\mathbf{L}^T \ (1A)$), on peut identifier les expressions mathématiques exactes permettant de calculer les coefficients de $\mathbf{D}$ et de $\mathbf{L}$ de manière itérative, colonne par colonne, de $j = 1$ à $N-1$.

#### 1. Calcul des termes diagonaux $D_j$

Par identification du coefficient diagonal $A_{jj}$ lors du produit matriciel, nous obtenons :
$$A_{jj} = \sum_{k=1}^{j} L_{jk} D_k L_{jk} = \sum_{k=1}^{j-1} L_{jk}^2 D_k + L_{jj}^2 D_j \tag{4A}$$

Puisque la matrice $\mathbf{L}$ est unitaire par définition, sa diagonale vaut $L_{jj} = 1$. En isolant $D_j$, on obtient la relation de récurrence :
$$D_j = A_{jj} - \sum_{k=1}^{j-1} L_{jk}^2 D_k \tag{5A}$$

#### 2. Calcul des termes extra-diagonaux $L_{ij}$ (pour les lignes $i > j$)

Pour un élément situé sous la diagonale dans la colonne $j$, le développement du produit donne :
$$A_{ij} = \sum_{k=1}^{j} L_{ik} D_k L_{jk} = \sum_{k=1}^{j-1} L_{ik} L_{jk} D_k + L_{ij} D_j L_{jj} \tag{7A}$$

En utilisant à nouveau le fait que $L_{jj} = 1$ et en isolant l'inconnue $L_{ij}$, on aboutit à la formule générale :
$$L_{ij} = \frac{1}{D_j} \left( A_{ij} - \sum_{k=1}^{j-1} L_{ik} L_{jk} D_k \right) \quad \text{pour } i = j+1, \dots, N-1 \tag{6A}$$

### C. Traduction algorithmique et optimisation vectorisée

L'implémentation NumPy utilise des tranches de tableaux et des produits matrice-vecteur pour réduire le coût des boucles interprétées. La factorisation dense reste d'ordre $O(n^3)$ ; la vectorisation ne change pas cet ordre.

L'extrait suivant illustre le calcul des coefficients diagonaux :


```python
for j in range(n):
    v = (L[j, :j] ** 2) * D_diag[:j]
    D_diag[j] = A[j, j] - np.sum(v)
```


2. **Calcul parallélisé de la sous-colonne de $L$ :** Pour toutes les lignes $i > j$ d'un seul coup, le terme $\sum_{k=1}^{j-1} L_{ik} L_{jk} D_k$ est traduit sous la forme d'un produit matrice-vecteur performant via l'opérateur @ :

```python
for j in range(n):
    if j < n - 1:
        somme_L = L[j+1:, :j] @ (L[j, :j] * D_diag[:j])
        L[j+1:, j] = (A[j+1:, j] - somme_L) / D_diag[j]
```


### D. Résolution du système linéaire global

Une fois la matrice Jacobienne factorisée ($\mathbf{J} = \mathbf{LDL}^T \ (1A)$), nous pouvons nous intéresser à la résolution du système linéaire tangent $\mathbf{J} \cdot \delta H = -F(H) \ (13)$ que l'on peut traduire ainsi :
$$(\mathbf{L} \cdot \mathbf{D} \cdot \mathbf{L}^T) \cdot x = b \tag{7A}$$

Pour résoudre ce problème complexe sans réaliser d'inversion matricielle directe (opération numériquement lourde et instable), nous introduisons deux changements de variables successifs (vecteurs intermédiaires $y$ et $z$) basés sur l'associativité du produit matriciel. Cela permet de segmenter le système en trois sous-systèmes simples.

#### Étape 1 : Introduction du vecteur intermédiaire $y$ et Descente

Par associativité, nous pouvons regrouper le produit à droite de $\mathbf{L}$ :
$$\mathbf{L} \cdot \underbrace{\left[ \mathbf{D} \cdot \mathbf{L}^T \cdot x \right]}_{\text{Posons } y} = b \Leftrightarrow y = \mathbf{D} \cdot \mathbf{L}^T \cdot x \tag{8A}$$

En effectuant le changement de variable $ (8A) $, le système global se réduit à un système triangulaire inférieur :
$$\mathbf{L} \cdot y = b \tag{9A}$$

Puisque $\mathbf{L}$ est triangulaire inférieure avec des $1$ sur sa diagonale, chaque composante $y_i$ se calcule explicitement et séquentiellement de la première à la dernière ligne par descente triangulaire (Forward Substitution) :
$${y_i = b_i - \sum_{k=1}^{i-1} L_{ik} y_k \quad \text{pour } i = 1, \dots, N-1} \tag{9A}$$

#### Étape 2 : Introduction du vecteur intermédiaire $z$ et Résolution Diagonale

Maintenant que le vecteur $y$ est entièrement connu, nous reprenons sa définition : $\mathbf{D} \cdot \mathbf{L}^T \cdot x = y \ (8A)$. Nous effectuons un second regroupement :
$$\mathbf{D} \cdot \underbrace{\left[ \mathbf{L}^T \cdot x \right]}_{\text{Posons } z} = y \Leftrightarrow z = \mathbf{L}^T \cdot x \tag{10A}$$

En posant $(10A)$, nous obtenons un système purement diagonal découplé :
$$\mathbf{D} \cdot z = y \tag{11A}$$

La résolution de ce système est immédiate et ne requiert qu'une simple division scalaire par élément diagonal :
$${z_i = \frac{y_i}{D_i} \quad \text{pour } i = 1, \dots, N-1} \tag{12A}$$

#### Étape 3 : Remontée triangulaire pour l'inconnue finale $x$

Il ne reste plus qu'à résoudre la dernière relation :
$$\mathbf{L}^T \cdot x = z \tag{13A}$$

La matrice $\mathbf{L}$ étant triangulaire inférieure, sa transposée $\mathbf{L}^T$ est triangulaire supérieure. Ses éléments diagonaux valent toujours $1$, et le coefficient à la ligne $i$ et colonne $k$ correspond à $L_{ki}$. Le vecteur inconnu final $x$ (représentant les corrections de potentiel $\delta H$ dans notre algorithme de Newton-Raphson) est calculé à rebours, de la dernière ligne à la première, par un algorithme de remontée triangulaire (Backward Substitution) :
$${x_i = z_i - \sum_{k=i+1}^{n} L_{ki} x_k \quad \text{pour } i = N-1, N-2, \dots, 1} \tag{14A}$$